# 🧠 Notebook 06 — LLM Narrative Engine
## BaliGuard — Early Warning System Krisis Pariwisata Bali

Membangun **LLM Narrative Engine** menggunakan **Claude API** yang mengubah output model menjadi laporan naratif otomatis Bahasa Indonesia.

**Input:** `data/final/predictions_final.csv`, `data/final/master_dataset_clean.parquet`  
**Output:** `src/narrative_engine.py`, `data/final/narratives_cache.json`

## 1. Import & Instalasi

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

try:
    import anthropic
    print('✅ anthropic SDK tersedia')
except ImportError:
    os.system('pip install anthropic -q')
    import anthropic
    print('✅ anthropic SDK berhasil diinstall')

print(f'Pandas: {pd.__version__}')

## 2. Konfigurasi API Key

In [ ]:
import os
from groq import Groq

# Set API key Groq — ganti string di bawah dengan key kamu
# Dapatkan gratis di: https://console.groq.com/keys

GROQ_API_KEY = os.getenv('GROQ_API_KEY', 'gsk_GANTI_DENGAN_API_KEY_KAMU')

if GROQ_API_KEY.startswith('gsk_GANTI'):
    print('⚠️  API key belum diset!')
    print('   Dapatkan gratis di: https://console.groq.com/keys')
else:
    client = Groq(api_key=GROQ_API_KEY)
    print('✅ Groq client siap')
    print(f'   Key: {GROQ_API_KEY[:12]}...')

## 3. Load Data & Model

In [ ]:
predictions = pd.read_csv('data/final/predictions_final.csv')
print('✅ predictions_final:', predictions.shape)
print('   Kolom:', predictions.columns.tolist())
print()

master = pd.read_parquet('data/final/master_dataset_clean.parquet')
print('✅ master_dataset_clean:', master.shape)
print()

scaler   = joblib.load('data/models/scaler.pkl')
rf_model = joblib.load('data/models/model_random_forest.pkl')
le       = joblib.load('data/models/label_encoder.pkl')
print('✅ Model artifacts loaded')
print()

print('=== 6 BULAN TERAKHIR ===')
print(predictions.tail(6)[['month','wisman','crisis_score_100',
    'crisis_level','rf_predicted_level','rf_confidence','iso_anomaly']].to_string())

## 4. Fungsi Build Context

In [ ]:
def build_crisis_context(month_str: str) -> dict:
    pred_row = predictions[predictions['month'] == month_str]
    if len(pred_row) == 0:
        pred_row = predictions.tail(1)
        month_str = pred_row['month'].values[0]
    pred_row = pred_row.iloc[0]

    idx = predictions[predictions['month'] == month_str].index[0]
    history = predictions.loc[max(0, idx-3):idx-1]

    ctx = {
        'month'        : month_str,
        'crisis_score' : round(float(pred_row['crisis_score_100']), 1),
        'crisis_level' : str(pred_row['crisis_level']),
        'rf_predicted' : str(pred_row['rf_predicted_level']),
        'rf_confidence': round(float(pred_row['rf_confidence']) * 100, 1),
        'is_anomaly'   : int(pred_row.get('iso_anomaly', 0)),
        'wisman'       : int(pred_row['wisman']),
        'tpk_bintang'  : round(float(pred_row.get('tpk_bintang', 0)), 1),
        'inflasi'      : round(float(pred_row.get('inflasi_processed', 0)), 2),
        'usd_idr'      : round(float(pred_row.get('usd_idr_avg', 0)), 0),
        'sentiment'    : round(float(pred_row.get('avg_sentiment_monthly', 0)), 3),
        'prob_krisis'  : round(float(pred_row.get('prob_krisis', 0)) * 100, 1),
        'prob_siaga'   : round(float(pred_row.get('prob_siaga', 0)) * 100, 1),
        'bali_share'   : round(float(pred_row.get('bali_share_pct', 0)), 1),
        'wisman_zscore': round(float(pred_row.get('wisman_zscore', 0)), 2),
    }
    if len(history) >= 2:
        avg3 = history['wisman'].mean()
        ctx['wisman_trend']  = 'naik' if ctx['wisman'] > avg3 else 'turun'
        ctx['avg_wisman_3m'] = round(avg3, 0)
        ctx['prev_levels']   = history['crisis_level'].tolist()
    else:
        ctx['wisman_trend']  = 'tidak tersedia'
        ctx['avg_wisman_3m'] = 0
        ctx['prev_levels']   = []
    return ctx

# Test
test_month = predictions['month'].iloc[-1]
ctx_test = build_crisis_context(test_month)
print(f'Konteks untuk: {test_month}')
for k, v in ctx_test.items():
    print(f'  {k:20s}: {v}')

## 5. Fungsi Build Prompt

In [ ]:
LEVEL_DESC = {
    'AMAN'    : 'kondisi pariwisata normal, tidak ada indikasi krisis',
    'WASPADA' : 'ada sinyal awal yang perlu dipantau',
    'SIAGA'   : 'tekanan signifikan pada sektor pariwisata',
    'KRISIS'  : 'krisis pariwisata yang membutuhkan respons segera'
}

def build_prompt(ctx: dict, report_type: str = 'summary') -> str:
    level_text = LEVEL_DESC.get(ctx['crisis_level'], ctx['crisis_level'])
    prev = ' → '.join(ctx['prev_levels']) if ctx['prev_levels'] else 'N/A'

    data_block = (
        f"DATA PARIWISATA BALI — {ctx['month']}\n"
        f"Crisis Score: {ctx['crisis_score']}/100 → Level {ctx['crisis_level']} ({level_text})\n"
        f"Prediksi RF: {ctx['rf_predicted']} (confidence: {ctx['rf_confidence']}%)\n"
        f"Anomali: {'Ya' if ctx['is_anomaly'] else 'Tidak'} | "
        f"P(Krisis): {ctx['prob_krisis']}% | P(Siaga): {ctx['prob_siaga']}%\n"
        f"Wisman: {ctx['wisman']:,.0f} (trend: {ctx['wisman_trend']}, "
        f"avg 3bln: {int(ctx['avg_wisman_3m']):,.0f})\n"
        f"Pangsa Bali/Nasional: {ctx['bali_share']}% | Z-score: {ctx['wisman_zscore']}\n"
        f"TPK Hotel: {ctx['tpk_bintang']}% | USD/IDR: Rp {int(ctx['usd_idr']):,}\n"
        f"Inflasi: {ctx['inflasi']}% | Sentimen: {ctx['sentiment']}\n"
        f"Histori 3 bulan: {prev}\n"
    )

    if report_type == 'summary':
        return (
            "Kamu adalah analis sistem BaliGuard (early warning pariwisata Bali).\n"
            + data_block
            + f"Buat ringkasan kondisi pariwisata Bali bulan {ctx['month']} "
            "dalam 2-3 kalimat Bahasa Indonesia. Tajam, berbasis data, "
            "cocok untuk KPI card dashboard."
        )
    elif report_type == 'alert':
        return (
            "Kamu adalah sistem BaliGuard. Kondisi kritis terdeteksi.\n"
            + data_block
            + "Buat PERINGATAN DARURAT (max 120 kata) Bahasa Indonesia: "
            "status level, top 3 indikator kritis, 1 rekomendasi tindakan segera. "
            "Tone profesional dan urgen."
        )
    else:
        return (
            "Kamu adalah analis BaliGuard.\n"
            + data_block
            + "Buat laporan bulanan Bahasa Indonesia:\n"
            "1. **Ringkasan Eksekutif** (2-3 kalimat): status dan signifikansi\n"
            "2. **Analisis Indikator** (3-4 kalimat): makna angka\n"
            "3. **Faktor Pendorong** (2-3 kalimat): penyebab kondisi\n"
            "4. **Rekomendasi** (3 poin konkret): untuk stakeholder\n"
            "Setiap klaim didukung angka dari data."
        )

print(f'Panjang prompt summary : {len(build_prompt(ctx_test, "summary"))} karakter')
print(f'Panjang prompt alert   : {len(build_prompt(ctx_test, "alert"))} karakter')
print(f'Panjang prompt monthly : {len(build_prompt(ctx_test, "monthly"))} karakter')

## 6. Fungsi Generate Narasi (Claude API)

In [ ]:
def generate_narrative(month_str: str, report_type: str = 'summary',
                       api_key: str = None) -> dict:
    if api_key is None:
        api_key = os.getenv('ANTHROPIC_API_KEY', '')
    if not api_key or api_key.startswith('sk-ant-GANTI'):
        return {'success': False,
                'narrative': '[API key belum dikonfigurasi. Set ANTHROPIC_API_KEY.]',
                'error': 'No API key'}
    try:
        ctx    = build_crisis_context(month_str)
        prompt = build_prompt(ctx, report_type)
        client = anthropic.Anthropic(api_key=api_key)
        msg    = client.messages.create(
            model='claude-sonnet-4-5', max_tokens=1024,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return {
            'success'      : True,
            'narrative'    : msg.content[0].text,
            'tokens'       : msg.usage.input_tokens + msg.usage.output_tokens,
            'month'        : month_str,
            'crisis_level' : ctx['crisis_level'],
            'report_type'  : report_type,
            'crisis_score' : ctx['crisis_score'],
        }
    except Exception as e:
        return {'success': False, 'narrative': f'[Error: {e}]', 'error': str(e)}

# Test
print(f'Testing generate untuk bulan: {test_month}')
result = generate_narrative(test_month, 'summary', ANTHROPIC_API_KEY)
print(f'Success: {result["success"]}')
if result['success']:
    print(f'Tokens : {result["tokens"]}')
    print()
    print('=== NARASI ===')
    print(result['narrative'])
else:
    print(f'Error  : {result.get("error")}')

## 7. Batch Generate — Semua Periode SIAGA & KRISIS

In [ ]:
crisis_months = predictions[
    predictions['crisis_level'].isin(['KRISIS', 'SIAGA'])
].sort_values('crisis_score_100', ascending=False)

print(f'Bulan KRISIS/SIAGA: {len(crisis_months)}')
narratives_cache = {}

if not ANTHROPIC_API_KEY.startswith('sk-ant-GANTI'):
    top_months = crisis_months.head(10)['month'].tolist()
    print(f'Generating {len(top_months)} narasi...')
    print()

    for i, month in enumerate(top_months, 1):
        level = predictions[predictions['month']==month]['crisis_level'].values[0]
        rtype = 'alert' if level == 'KRISIS' else 'monthly'
        print(f'  [{i:2d}/{len(top_months)}] {month} ({level})...', end='', flush=True)
        result = generate_narrative(month, rtype, ANTHROPIC_API_KEY)
        if result['success']:
            narratives_cache[month] = result
            print(f' ✅ {result["tokens"]} tokens')
        else:
            print(f' ❌ {result["error"]}')

    os.makedirs('data/final', exist_ok=True)
    with open('data/final/narratives_cache.json', 'w', encoding='utf-8') as f:
        json.dump(narratives_cache, f, ensure_ascii=False, indent=2)
    print(f'\n✅ narratives_cache.json disimpan ({len(narratives_cache)} narasi)')
else:
    print('⚠️  API key belum diset — skip batch generation')

## 8. Export `src/narrative_engine.py`

In [ ]:
os.makedirs('src', exist_ok=True)

engine_code = 'import os\nimport json\nimport anthropic\nimport numpy as np\n\nLEVEL_DESC = {\n    \'AMAN\'    : \'kondisi pariwisata normal, tidak ada indikasi krisis\',\n    \'WASPADA\' : \'ada sinyal awal yang perlu dipantau\',\n    \'SIAGA\'   : \'tekanan signifikan pada sektor pariwisata\',\n    \'KRISIS\'  : \'krisis pariwisata yang membutuhkan respons segera\'\n}\n\ndef build_context(pred_row: dict, history_rows: list = None) -> dict:\n    ctx = {\n        \'month\'        : str(pred_row.get(\'month\', \'N/A\')),\n        \'crisis_score\' : round(float(pred_row.get(\'crisis_score_100\', 0)), 1),\n        \'crisis_level\' : str(pred_row.get(\'crisis_level\', \'WASPADA\')),\n        \'rf_predicted\' : str(pred_row.get(\'rf_predicted_level\', \'N/A\')),\n        \'rf_confidence\': round(float(pred_row.get(\'rf_confidence\', 0)) * 100, 1),\n        \'is_anomaly\'   : int(pred_row.get(\'iso_anomaly\', 0)),\n        \'wisman\'       : int(pred_row.get(\'wisman\', 0)),\n        \'tpk_bintang\'  : round(float(pred_row.get(\'tpk_bintang\', 0)), 1),\n        \'inflasi\'      : round(float(pred_row.get(\'inflasi_processed\', 0)), 2),\n        \'usd_idr\'      : round(float(pred_row.get(\'usd_idr_avg\', 0)), 0),\n        \'sentiment\'    : round(float(pred_row.get(\'avg_sentiment_monthly\', 0)), 3),\n        \'prob_krisis\'  : round(float(pred_row.get(\'prob_krisis\', 0)) * 100, 1),\n        \'prob_siaga\'   : round(float(pred_row.get(\'prob_siaga\', 0)) * 100, 1),\n        \'bali_share\'   : round(float(pred_row.get(\'bali_share_pct\', 0)), 1),\n        \'wisman_zscore\': round(float(pred_row.get(\'wisman_zscore\', 0)), 2),\n    }\n    if history_rows and len(history_rows) >= 2:\n        avg3 = np.mean([r.get(\'wisman\', 0) for r in history_rows[-3:]])\n        ctx[\'wisman_trend\']  = \'naik\' if ctx[\'wisman\'] > avg3 else \'turun\'\n        ctx[\'avg_wisman_3m\'] = round(avg3, 0)\n        ctx[\'prev_levels\']   = [r.get(\'crisis_level\', \'N/A\') for r in history_rows[-3:]]\n    else:\n        ctx[\'wisman_trend\']  = \'tidak tersedia\'\n        ctx[\'avg_wisman_3m\'] = 0\n        ctx[\'prev_levels\']   = []\n    return ctx\n\ndef build_prompt(ctx: dict, report_type: str = \'summary\') -> str:\n    level_text = LEVEL_DESC.get(ctx[\'crisis_level\'], ctx[\'crisis_level\'])\n    prev = \' -> \'.join(ctx[\'prev_levels\']) if ctx[\'prev_levels\'] else \'N/A\'\n    data_block = (\n        f"DATA PARIWISATA BALI - {ctx[\'month\']}\\n"\n        f"Crisis Score: {ctx[\'crisis_score\']}/100 -> Level {ctx[\'crisis_level\']} ({level_text})\\n"\n        f"Prediksi RF: {ctx[\'rf_predicted\']} (confidence: {ctx[\'rf_confidence\']}%)\\n"\n        f"Anomali: {\'Ya\' if ctx[\'is_anomaly\'] else \'Tidak\'} | P(Krisis): {ctx[\'prob_krisis\']}% | P(Siaga): {ctx[\'prob_siaga\']}%\\n"\n        f"Wisman: {ctx[\'wisman\']:,.0f} (trend: {ctx[\'wisman_trend\']}, avg 3bln: {int(ctx[\'avg_wisman_3m\']):,.0f})\\n"\n        f"Pangsa Bali: {ctx[\'bali_share\']}% | Z-score: {ctx[\'wisman_zscore\']}\\n"\n        f"TPK Hotel: {ctx[\'tpk_bintang\']}% | USD/IDR: Rp {int(ctx[\'usd_idr\']):,}\\n"\n        f"Inflasi: {ctx[\'inflasi\']}% | Sentimen: {ctx[\'sentiment\']}\\n"\n        f"Histori: {prev}\\n"\n    )\n    if report_type == \'summary\':\n        return ("Kamu adalah analis BaliGuard (early warning pariwisata Bali).\\n"\n                + data_block\n                + f"Buat ringkasan kondisi pariwisata Bali bulan {ctx[\'month\']} "\n                "dalam 2-3 kalimat Bahasa Indonesia. Tajam, berbasis data, cocok untuk KPI card.")\n    elif report_type == \'alert\':\n        return ("Kamu adalah sistem BaliGuard. Kondisi kritis terdeteksi.\\n"\n                + data_block\n                + "Buat PERINGATAN DARURAT (max 120 kata) Bahasa Indonesia: "\n                "status level, top 3 indikator kritis, 1 rekomendasi tindakan segera.")\n    else:\n        return ("Kamu adalah analis BaliGuard.\\n"\n                + data_block\n                + "Buat laporan bulanan Bahasa Indonesia:\\n"\n                "1. Ringkasan Eksekutif (2-3 kalimat)\\n"\n                "2. Analisis Indikator (3-4 kalimat)\\n"\n                "3. Faktor Pendorong (2-3 kalimat)\\n"\n                "4. Rekomendasi (3 poin konkret)")\n\ndef generate(pred_row: dict, report_type: str = \'summary\',\n             api_key: str = None, history_rows: list = None) -> dict:\n    if api_key is None:\n        api_key = os.getenv(\'ANTHROPIC_API_KEY\', \'\')\n    if not api_key:\n        return {\'success\': False, \'narrative\': \'[API key tidak dikonfigurasi]\', \'error\': \'No API key\'}\n    try:\n        ctx = build_context(pred_row, history_rows)\n        client = anthropic.Anthropic(api_key=api_key)\n        msg = client.messages.create(\n            model=\'claude-sonnet-4-5\', max_tokens=1024,\n            messages=[{\'role\': \'user\', \'content\': build_prompt(ctx, report_type)}]\n        )\n        return {\n            \'success\'      : True,\n            \'narrative\'    : msg.content[0].text,\n            \'tokens\'       : msg.usage.input_tokens + msg.usage.output_tokens,\n            \'month\'        : ctx[\'month\'],\n            \'crisis_level\' : ctx[\'crisis_level\'],\n        }\n    except Exception as e:\n        return {\'success\': False, \'narrative\': f\'[Error: {e}]\', \'error\': str(e)}\n\ndef load_cache(cache_path: str = \'data/final/narratives_cache.json\') -> dict:\n    if os.path.exists(cache_path):\n        with open(cache_path, \'r\', encoding=\'utf-8\') as f:\n            return json.load(f)\n    return {}\n'

with open('src/__init__.py', 'w') as f:
    f.write('')

with open('src/narrative_engine.py', 'w', encoding='utf-8') as f:
    f.write(engine_code)

print('✅ src/narrative_engine.py disimpan')
print('✅ src/__init__.py disimpan')
print()
print('Import di dashboard:')
print('  from src.narrative_engine import generate, build_context, load_cache')

## 9. Checklist File

In [ ]:
print('=' * 55)
print('  BALIGUARD — RINGKASAN NB06')
print('=' * 55)
print()
required = [
    'data/final/master_dataset_clean.parquet',
    'data/final/predictions_final.csv',
    'data/final/narratives_cache.json',
    'data/models/model_random_forest.pkl',
    'data/models/model_isolation_forest.pkl',
    'data/models/scaler.pkl',
    'data/models/label_encoder.pkl',
    'src/narrative_engine.py',
]
for f in required:
    status = '✅' if os.path.exists(f) else '❌ belum ada'
    print(f'   {status} {f}')
print()
print('Jalankan dashboard:')
print('  pip install streamlit plotly anthropic pyarrow joblib')
print('  streamlit run dashboard.py')
print('=' * 55)